# 🧬 Dark Manifold V19 — The Complete Fix

## Three Problems → Three Solutions

```
┌─────────────────────────────────────────────────────────────────────┐
│                    V19: FIXING EVERYTHING                           │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   PROBLEM 1: Model doesn't know which stress is applied             │
│   FIX 1: Perturbation encoding as explicit input                    │
│                                                                     │
│   PROBLEM 2: Testing extrapolation instead of interpolation         │
│   FIX 2: Multiple replicates per condition, proper train/test split │
│                                                                     │
│   PROBLEM 3: No adaptation mechanism                                │
│   FIX 3: Context encoder that reads early timepoints                │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

## V19 Architecture

```
                    ┌─────────────────┐
    Perturbation →  │  Perturbation   │ → stress_embedding
    [temp, H2O2,    │    Encoder      │    (learned representation)
     glucose, ...]  └─────────────────┘
                            │
                            ▼
    ┌─────────────┐   ┌─────────────────┐   ┌─────────────┐
    │  Context    │   │                 │   │             │
    │  Encoder    │ → │   Metabolic     │ → │  Trajectory │
    │ (first 3    │   │   Dynamics      │   │  Prediction │
    │  timepoints)│   │   Network       │   │             │
    └─────────────┘   └─────────────────┘   └─────────────┘
          ↑                   ↑
    Early data          Metabolite state
```

## Test Protocol

1. **Within-condition test**: Train on replicates 1-8, test on replicates 9-10
2. **Cross-condition test**: Train on 5 conditions, test on held-out 6th
3. **Interpolation test**: Train on temp=[42°C, 48°C], test on temp=45°C

In [ ]:
#@title 1️⃣ Setup
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import time
import json
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Device: {device}")
if device.type == 'cuda':
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")

torch.manual_seed(42)
np.random.seed(42)

In [ ]:
#@title 2️⃣ Configuration

# Model architecture
HIDDEN_DIM = 256
N_REACTIONS = 100
PERTURB_DIM = 5      # Dimension of perturbation vector
CONTEXT_DIM = 64     # Context encoder output
N_CONTEXT_STEPS = 3  # How many early timepoints for context

# Training
N_EPOCHS = 1500
LEARNING_RATE = 1e-3
DT = 0.02

# Data
N_REPLICATES = 10    # Replicates per condition
TRAIN_REPLICATES = 8 # Use 8 for training
TEST_REPLICATES = 2  # Use 2 for testing

print("📋 V19 Configuration:")
print(f"   Perturbation encoding: {PERTURB_DIM}D vector")
print(f"   Context encoder: {N_CONTEXT_STEPS} early timepoints → {CONTEXT_DIM}D")
print(f"   Replicates: {TRAIN_REPLICATES} train / {TEST_REPLICATES} test per condition")
print(f"   Training: {N_EPOCHS} epochs")

In [ ]:
#@title 3️⃣ FIX 1: Perturbation Encoding
#@markdown Each stress condition is encoded as a continuous vector
#@markdown representing the physical/chemical perturbation.

# Perturbation vectors: [temp_delta, H2O2, glucose, lactose, growth_rate]
# Normalized to roughly [-1, 1] range

PERTURBATION_VECTORS = {
    'heat_shock': {
        'vector': [0.8, 0.0, 1.0, 0.0, 0.8],   # +8°C, normal glucose, slightly reduced growth
        'description': 'Temperature +8°C above optimal'
    },
    'cold_shock': {
        'vector': [-0.8, 0.0, 1.0, 0.0, 0.3],  # -21°C (scaled), normal glucose, low growth
        'description': 'Temperature -21°C below optimal'
    },
    'oxidative_stress': {
        'vector': [0.0, 1.0, 1.0, 0.0, 0.5],   # H2O2 present, normal glucose
        'description': 'H2O2 oxidative stress'
    },
    'diauxic_shift': {
        'vector': [0.0, 0.0, -0.5, 1.0, 0.5],  # Glucose depleting, lactose appearing
        'description': 'Glucose → Lactose transition'
    },
    'stationary_phase': {
        'vector': [0.0, 0.0, -1.0, 0.0, 0.1],  # No glucose, minimal growth
        'description': 'Nutrient depletion, growth arrest'
    },
    'carbon_variation': {
        'vector': [0.0, 0.0, 0.0, 0.0, 0.6],   # Mixed carbon, moderate growth
        'description': 'Acetate/glycerol carbon sources'
    }
}

# For interpolation test: create intermediate perturbations
PERTURBATION_INTERPOLATIONS = {
    'mild_heat': {
        'vector': [0.4, 0.0, 1.0, 0.0, 0.9],   # +4°C
        'description': 'Mild heat stress (+4°C)'
    },
    'severe_heat': {
        'vector': [1.0, 0.0, 1.0, 0.0, 0.5],   # +12°C
        'description': 'Severe heat stress (+12°C)'
    },
    'mild_oxidative': {
        'vector': [0.0, 0.5, 1.0, 0.0, 0.7],   # Low H2O2
        'description': 'Mild oxidative stress'
    },
    'combined_heat_oxidative': {
        'vector': [0.5, 0.5, 1.0, 0.0, 0.5],   # Heat + H2O2
        'description': 'Combined heat and oxidative stress'
    }
}

print("✅ FIX 1: Perturbation Encoding")
print("\n📊 Perturbation Vectors [temp, H2O2, glucose, lactose, growth]:")
for name, data in PERTURBATION_VECTORS.items():
    print(f"   {name:20s}: {data['vector']}")

print("\n🔬 Interpolation Test Vectors:")
for name, data in PERTURBATION_INTERPOLATIONS.items():
    print(f"   {name:20s}: {data['vector']}")

In [ ]:
#@title 4️⃣ FIX 2: Multiple Replicates with Proper Train/Test Split
#@markdown Generate 10 replicates per condition with biological variation.
#@markdown Train on 8, test on 2.

def create_replicate_datasets(n_replicates=10):
    """Create multiple replicates per condition with realistic variation."""
    
    metabolites = [
        'glucose-6-P', 'fructose-6-P', 'fructose-1,6-BP', 'DHAP', 'G3P',
        '3-PGA', 'PEP', 'pyruvate',
        'citrate', 'isocitrate', 'alpha-KG', 'succinate', 'fumarate', 'malate', 'oxaloacetate',
        '6-P-gluconate', 'ribose-5-P', 'ribulose-5-P', 'E4P',
        'alanine', 'arginine', 'asparagine', 'aspartate', 'cysteine',
        'glutamate', 'glutamine', 'glycine', 'histidine', 'isoleucine',
        'leucine', 'lysine', 'methionine', 'phenylalanine', 'proline',
        'serine', 'threonine', 'tryptophan', 'tyrosine', 'valine',
        'ATP', 'ADP', 'AMP', 'GTP', 'GDP', 'UTP', 'CTP',
        'NAD', 'NADH', 'NADP', 'NADPH', 'CoA', 'acetyl-CoA',
        'trehalose', 'glycerol', 'lactate', 'acetate', 'formate',
        'cAMP', 'ppGpp', 'homoserine', 'putrescine', 'spermidine'
    ]
    
    n_mets = len(metabolites)
    times = np.array([0, 10, 20, 30, 40, 50, 60, 90, 120, 150, 180]) / 60.0
    n_times = len(times)
    
    aa_names = ['alanine', 'arginine', 'asparagine', 'aspartate', 'cysteine',
                'glutamate', 'glutamine', 'glycine', 'histidine', 'isoleucine',
                'leucine', 'lysine', 'methionine', 'phenylalanine', 'proline',
                'serine', 'threonine', 'tryptophan', 'tyrosine', 'valine']
    aa_indices = [metabolites.index(aa) for aa in aa_names]
    
    def add_biological_variation(base_data, replicate_idx, condition):
        """Add realistic biological variation to a replicate."""
        # Different types of variation:
        # 1. Amplitude variation (±20%)
        amp_var = 1.0 + 0.2 * np.random.randn()
        
        # 2. Timing variation (±10% shift in dynamics)
        time_var = 1.0 + 0.1 * np.random.randn()
        
        # 3. Baseline variation (±10%)
        baseline_var = 1.0 + 0.1 * np.random.randn(n_mets)
        
        # 4. Measurement noise (±5%)
        noise = 0.05 * np.random.randn(*base_data.shape)
        
        # Apply variations
        varied_data = base_data.copy()
        varied_data = (varied_data - 1.0) * amp_var + 1.0  # Amplitude
        varied_data = varied_data * baseline_var  # Baseline
        varied_data = varied_data + noise  # Noise
        
        return np.clip(varied_data, 0.05, 15.0)
    
    def gen_base_heat():
        data = np.ones((n_times, n_mets))
        for i, met in enumerate(metabolites):
            if met in ['glucose-6-P', 'fructose-6-P', 'fructose-1,6-BP']:
                data[:, i] = 1.0 - 0.5 * (1 - np.exp(-times * 3))
            elif met in aa_names:
                data[:, i] = 1.0 + 0.6 * np.exp(-(times - 0.5)**2 / 0.3)
            elif met == 'trehalose':
                data[:, i] = 1.0 + 2.0 * (1 - np.exp(-times * 1))
            elif met == 'ATP':
                data[:, i] = 1.0 - 0.3 * (1 - np.exp(-times * 4))
            elif met == 'ppGpp':
                data[:, i] = 1.0 + 1.5 * np.exp(-times / 0.5) * times
        return data
    
    def gen_base_cold():
        data = np.ones((n_times, n_mets))
        for i, met in enumerate(metabolites):
            if met in ['glucose-6-P', 'fructose-6-P']:
                data[:, i] = 1.0 + 0.4 * (1 - np.exp(-times * 2))
            elif met in aa_names:
                data[:, i] = 1.0 + 0.5 * (1 - np.exp(-times * 1.5))
            elif met in ['ATP', 'GTP']:
                data[:, i] = 1.0 + 0.3 * (1 - np.exp(-times * 1))
            elif met == 'trehalose':
                data[:, i] = 1.0 + 0.8 * (1 - np.exp(-times * 0.5))
        return data
    
    def gen_base_oxidative():
        data = np.ones((n_times, n_mets))
        for i, met in enumerate(metabolites):
            if met in ['glucose-6-P', '3-PGA']:
                data[:, i] = 1.0 - 0.6 * (1 - np.exp(-times * 5))
            elif met in ['6-P-gluconate', 'ribose-5-P']:
                data[:, i] = 1.0 + 1.2 * (1 - np.exp(-times * 2))
            elif met == 'NADPH':
                data[:, i] = 0.4 + 0.6 * (1 - np.exp(-times * 2))
            elif met in aa_names:
                data[:, i] = 1.0 + 0.2 * np.sin(times * 3) * np.exp(-times * 0.5)
            elif met == 'glutamate':
                data[:, i] = 1.0 - 0.3 * (1 - np.exp(-times * 2))  # Consumed for glutathione
        return data
    
    def gen_base_diauxic():
        data = np.ones((n_times, n_mets))
        shift = 0.5
        for i, met in enumerate(metabolites):
            if met == 'PEP':
                data[:, i] = 1.0 + 2.5 * np.exp(-(times - shift)**2 / 0.1)
            elif met in ['glucose-6-P', 'fructose-6-P']:
                data[:, i] = np.where(times < shift, 1.0, 0.2 + 0.5 * (times - shift))
            elif met == 'cAMP':
                data[:, i] = 1.0 + 4.0 / (1 + np.exp(-15 * (times - shift)))
            elif met == 'lactate':
                data[:, i] = 0.1 + 1.5 * np.maximum(0, times - shift)
            elif met in aa_names:
                data[:, i] = 1.0 - 0.3 * np.exp(-(times - shift)**2 / 0.2) + 0.15 * times
        return data
    
    def gen_base_stationary():
        data = np.ones((n_times, n_mets))
        for i, met in enumerate(metabolites):
            if met in ['glucose-6-P', 'fructose-1,6-BP', 'pyruvate']:
                data[:, i] = 1.0 - 0.5 * (1 - np.exp(-times * 0.8))
            elif met in aa_names:
                data[:, i] = 1.0 + 1.2 * (1 - np.exp(-times * 0.5))
            elif met == 'ppGpp':
                data[:, i] = 1.0 + 3.5 * (1 - np.exp(-times * 1))
            elif met == 'trehalose':
                data[:, i] = 1.0 + 2.5 * (1 - np.exp(-times * 0.5))
            elif met == 'ATP':
                data[:, i] = 1.0 - 0.35 * (1 - np.exp(-times * 0.3))
        return data
    
    def gen_base_carbon():
        data = np.ones((n_times, n_mets))
        for i, met in enumerate(metabolites):
            if met == 'fructose-1,6-BP':
                data[:, i] = 1.8 * np.exp(-times * 1.5) + 0.3
            elif met in ['citrate', 'alpha-KG', 'malate']:
                data[:, i] = 1.0 + 0.8 * (1 - np.exp(-times * 0.8))
            elif met == 'PEP':
                data[:, i] = 1.0 + 0.4 * times
            elif met in aa_names:
                data[:, i] = 1.0 + 0.15 * np.sin(times * 2)
            elif met == 'acetyl-CoA':
                data[:, i] = 1.0 + 1.2 * (1 - np.exp(-times * 0.5))
        return data
    
    # Generate base trajectories
    base_generators = {
        'heat_shock': gen_base_heat,
        'cold_shock': gen_base_cold,
        'oxidative_stress': gen_base_oxidative,
        'diauxic_shift': gen_base_diauxic,
        'stationary_phase': gen_base_stationary,
        'carbon_variation': gen_base_carbon
    }
    
    # Create dataset with replicates
    datasets = {}
    
    for cond_name, gen_func in base_generators.items():
        base_data = gen_func()
        replicates = []
        
        for rep_idx in range(n_replicates):
            rep_data = add_biological_variation(base_data, rep_idx, cond_name)
            replicates.append(rep_data)
        
        datasets[cond_name] = {
            'replicates': np.array(replicates),  # (n_replicates, n_times, n_mets)
            'times': times,
            'metabolites': metabolites,
            'aa_indices': aa_indices,
            'perturbation': PERTURBATION_VECTORS[cond_name]['vector']
        }
    
    return datasets, metabolites, aa_indices

# Create datasets
all_datasets, metabolite_names, aa_indices = create_replicate_datasets(N_REPLICATES)
n_mets = len(metabolite_names)

print("\n✅ FIX 2: Multiple Replicates with Train/Test Split")
print(f"\n📊 Dataset Structure:")
for cond, ds in all_datasets.items():
    shape = ds['replicates'].shape
    print(f"   {cond:20s}: {shape[0]} replicates × {shape[1]} timepoints × {shape[2]} metabolites")

print(f"\n🔀 Split: {TRAIN_REPLICATES} train / {TEST_REPLICATES} test per condition")

In [ ]:
#@title 5️⃣ FIX 3: V19 Model with Context Encoder
#@markdown The model now has:
#@markdown 1. Perturbation encoder (knows the stress type)
#@markdown 2. Context encoder (adapts from early timepoints)
#@markdown 3. Combined dynamics predictor

class ContextEncoder(nn.Module):
    """Encodes early timepoints to extract condition-specific context."""
    
    def __init__(self, n_mets, n_context_steps, context_dim):
        super().__init__()
        self.n_context_steps = n_context_steps
        
        # Process sequence of metabolite states
        self.encoder = nn.Sequential(
            nn.Linear(n_mets * n_context_steps, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Linear(128, context_dim)
        )
    
    def forward(self, early_trajectory):
        """
        Args:
            early_trajectory: (batch, n_context_steps, n_mets)
        Returns:
            context: (batch, context_dim)
        """
        batch_size = early_trajectory.shape[0]
        flat = early_trajectory.reshape(batch_size, -1)
        return self.encoder(flat)


class PerturbationEncoder(nn.Module):
    """Encodes the perturbation vector into a learned embedding."""
    
    def __init__(self, perturb_dim, embed_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(perturb_dim, 64),
            nn.GELU(),
            nn.Linear(64, embed_dim)
        )
    
    def forward(self, perturbation):
        return self.encoder(perturbation)


class DarkManifoldV19(nn.Module):
    """
    V19: Complete fix with perturbation encoding + context adaptation.
    
    The model knows:
    1. What stress is applied (perturbation vector)
    2. How this specific experiment is behaving (context from early timepoints)
    """
    
    def __init__(self, n_mets, n_reactions, hidden_dim, perturb_dim, context_dim, n_context_steps):
        super().__init__()
        self.n_mets = n_mets
        self.n_reactions = n_reactions
        self.n_context_steps = n_context_steps
        
        # Encoders
        self.perturbation_encoder = PerturbationEncoder(perturb_dim, 64)
        self.context_encoder = ContextEncoder(n_mets, n_context_steps, context_dim)
        
        # Combined condition embedding
        self.condition_combiner = nn.Sequential(
            nn.Linear(64 + context_dim, 128),
            nn.GELU(),
            nn.Linear(128, 64)
        )
        
        # Stoichiometry (learned)
        self.S = nn.Parameter(torch.randn(n_mets, n_reactions) * 0.1)
        
        # Flux network: metabolites + condition → fluxes
        self.flux_net = nn.Sequential(
            nn.Linear(n_mets + 64, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, n_reactions)
        )
        
        # Condition-modulated degradation
        self.deg_net = nn.Sequential(
            nn.Linear(n_mets + 64, hidden_dim // 2),
            nn.GELU(),
            nn.Linear(hidden_dim // 2, n_mets),
            nn.Softplus()
        )
        
        # Regulation network
        self.reg_net = nn.Sequential(
            nn.Linear(n_mets + 64, hidden_dim // 2),
            nn.GELU(),
            nn.Linear(hidden_dim // 2, n_mets),
            nn.Tanh()
        )
        
        # Homeostasis (condition-dependent baseline)
        self.baseline_net = nn.Sequential(
            nn.Linear(64, n_mets),
            nn.Softplus()
        )
        self.homeo_strength = nn.Parameter(torch.tensor(0.05))
    
    def get_condition_embedding(self, perturbation, early_trajectory):
        """Combine perturbation and context into single condition embedding."""
        perturb_emb = self.perturbation_encoder(perturbation)  # (batch, 64)
        context_emb = self.context_encoder(early_trajectory)    # (batch, context_dim)
        combined = torch.cat([perturb_emb, context_emb], dim=-1)
        return self.condition_combiner(combined)  # (batch, 64)
    
    def forward(self, M, condition_emb, dt):
        """Single dynamics step."""
        # Concatenate metabolites with condition
        M_cond = torch.cat([M, condition_emb], dim=-1)
        
        # Fluxes
        v = self.flux_net(M_cond)
        stoich = torch.matmul(v, self.S.T)
        
        # Degradation (condition-dependent)
        deg_rates = self.deg_net(M_cond).clamp(0.01, 0.5)
        deg = deg_rates * M
        
        # Regulation
        reg = self.reg_net(M_cond) * M * 0.2
        
        # Homeostasis toward condition-specific baseline
        baseline = self.baseline_net(condition_emb) + 0.5  # Ensure positive
        homeo = self.homeo_strength.clamp(0.01, 0.2) * (baseline - M)
        
        # Total
        dM = stoich - deg + reg + homeo
        M_new = torch.clamp(M + dt * dM, 0.01, 15.0)
        
        return M_new
    
    def simulate(self, M0, perturbation, early_trajectory, times, dt=0.01):
        """
        Simulate full trajectory.
        
        Args:
            M0: Initial metabolites (batch, n_mets)
            perturbation: Perturbation vector (batch, perturb_dim)
            early_trajectory: First few timepoints (batch, n_context_steps, n_mets)
            times: Target timepoints
            dt: Integration step
        """
        # Get condition embedding once
        condition_emb = self.get_condition_embedding(perturbation, early_trajectory)
        
        # Simulate
        traj = [M0]
        M = M0.clone()
        t = 0.0
        
        for target in times[1:]:
            while t < target - 1e-6:
                step = min(dt, target - t)
                M = self.forward(M, condition_emb, step)
                t += step
            traj.append(M.clone())
        
        return torch.stack(traj, dim=1)  # (batch, n_times, n_mets)


print("✅ FIX 3: V19 Model with Context Encoder")
model = DarkManifoldV19(
    n_mets=n_mets,
    n_reactions=N_REACTIONS,
    hidden_dim=HIDDEN_DIM,
    perturb_dim=PERTURB_DIM,
    context_dim=CONTEXT_DIM,
    n_context_steps=N_CONTEXT_STEPS
).to(device)

print(f"\n📊 Model Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"   Perturbation encoder: {sum(p.numel() for p in model.perturbation_encoder.parameters()):,}")
print(f"   Context encoder:      {sum(p.numel() for p in model.context_encoder.parameters()):,}")
print(f"   Dynamics network:     {sum(p.numel() for p in model.flux_net.parameters()):,}")

In [ ]:
#@title 6️⃣ Training Functions

def prepare_batch(datasets, condition_names, replicate_indices, device):
    """Prepare a training batch from multiple conditions and replicates."""
    batch_data = []
    batch_perturb = []
    batch_times = None
    
    for cond_name in condition_names:
        ds = datasets[cond_name]
        for rep_idx in replicate_indices:
            if rep_idx < ds['replicates'].shape[0]:
                batch_data.append(ds['replicates'][rep_idx])
                batch_perturb.append(ds['perturbation'])
                if batch_times is None:
                    batch_times = ds['times']
    
    data = torch.tensor(np.array(batch_data), dtype=torch.float32, device=device)
    perturb = torch.tensor(np.array(batch_perturb), dtype=torch.float32, device=device)
    times = torch.tensor(batch_times, dtype=torch.float32, device=device)
    
    return data, perturb, times


def train_epoch(model, datasets, train_conditions, train_replicates, optimizer, dt, device):
    """Train one epoch on all training data."""
    model.train()
    total_loss = 0.0
    n_batches = 0
    
    for cond_name in train_conditions:
        ds = datasets[cond_name]
        perturb = torch.tensor([ds['perturbation']], dtype=torch.float32, device=device)
        times = ds['times']
        times_t = torch.tensor(times, dtype=torch.float32, device=device)
        
        for rep_idx in train_replicates:
            data = torch.tensor(ds['replicates'][rep_idx:rep_idx+1], dtype=torch.float32, device=device)
            
            # Get initial state and early trajectory for context
            M0 = data[:, 0, :]
            early_traj = data[:, :N_CONTEXT_STEPS, :]
            
            # Forward pass
            pred = model.simulate(M0, perturb, early_traj, times_t, dt=dt)
            
            # Loss: MSE + log-MSE + derivative matching
            loss_mse = F.mse_loss(pred, data)
            loss_log = F.mse_loss(torch.log(pred + 0.01), torch.log(data + 0.01))
            
            # Derivative loss (trajectory shape)
            diff_pred = pred[:, 1:, :] - pred[:, :-1, :]
            diff_true = data[:, 1:, :] - data[:, :-1, :]
            loss_deriv = F.mse_loss(diff_pred, diff_true)
            
            loss = loss_mse + 0.5 * loss_log + 0.3 * loss_deriv
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            total_loss += loss.item()
            n_batches += 1
    
    return total_loss / max(n_batches, 1)


def evaluate(model, datasets, conditions, replicate_indices, dt, device):
    """Evaluate model on given conditions and replicates."""
    model.eval()
    results = {}
    
    with torch.no_grad():
        for cond_name in conditions:
            ds = datasets[cond_name]
            perturb = torch.tensor([ds['perturbation']], dtype=torch.float32, device=device)
            times = ds['times']
            times_t = torch.tensor(times, dtype=torch.float32, device=device)
            
            cond_corrs = []
            cond_aa_corrs = []
            
            for rep_idx in replicate_indices:
                if rep_idx >= ds['replicates'].shape[0]:
                    continue
                    
                data = torch.tensor(ds['replicates'][rep_idx:rep_idx+1], dtype=torch.float32, device=device)
                
                M0 = data[:, 0, :]
                early_traj = data[:, :N_CONTEXT_STEPS, :]
                
                pred = model.simulate(M0, perturb, early_traj, times_t, dt=dt)
                
                pred_np = pred[0].cpu().numpy()
                true_np = data[0].cpu().numpy()
                
                # Overall correlation
                corr = np.corrcoef(pred_np.flatten(), true_np.flatten())[0, 1]
                if not np.isnan(corr):
                    cond_corrs.append(corr)
                
                # AA correlation
                aa_corrs = []
                for aa_idx in ds['aa_indices']:
                    if np.std(true_np[:, aa_idx]) > 0.01:
                        c = np.corrcoef(pred_np[:, aa_idx], true_np[:, aa_idx])[0, 1]
                        if not np.isnan(c):
                            aa_corrs.append(c)
                if aa_corrs:
                    cond_aa_corrs.append(np.mean(aa_corrs))
            
            results[cond_name] = {
                'overall': np.mean(cond_corrs) if cond_corrs else 0.0,
                'aa_avg': np.mean(cond_aa_corrs) if cond_aa_corrs else 0.0,
                'n_replicates': len(cond_corrs)
            }
    
    return results

print("✅ Training functions defined")

In [ ]:
#@title 7️⃣ Run Training 🚀

print("="*70)
print("V19 TRAINING: All Three Fixes Applied")
print("="*70)

# Define train/test split
TRAIN_CONDITIONS = ['heat_shock', 'cold_shock', 'oxidative_stress', 'diauxic_shift', 'stationary_phase']
TEST_CONDITIONS = ['carbon_variation']  # Held-out condition

TRAIN_REP_INDICES = list(range(TRAIN_REPLICATES))  # 0-7
TEST_REP_INDICES = list(range(TRAIN_REPLICATES, N_REPLICATES))  # 8-9

print(f"\n📚 Training Conditions: {TRAIN_CONDITIONS}")
print(f"🧪 Test Condition (held-out): {TEST_CONDITIONS}")
print(f"📊 Replicates: Train on {TRAIN_REP_INDICES}, Test on {TEST_REP_INDICES}")

# Initialize
model = DarkManifoldV19(
    n_mets=n_mets,
    n_reactions=N_REACTIONS,
    hidden_dim=HIDDEN_DIM,
    perturb_dim=PERTURB_DIM,
    context_dim=CONTEXT_DIM,
    n_context_steps=N_CONTEXT_STEPS
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)

# Training loop
losses = []
eval_history = []

print(f"\n🚀 Starting training for {N_EPOCHS} epochs...\n")
start_time = time.time()

for epoch in tqdm(range(N_EPOCHS)):
    loss = train_epoch(model, all_datasets, TRAIN_CONDITIONS, TRAIN_REP_INDICES, 
                       optimizer, DT, device)
    losses.append(loss)
    scheduler.step()
    
    # Evaluate every 100 epochs
    if (epoch + 1) % 100 == 0 or epoch == 0:
        # Evaluate on train conditions (train replicates)
        train_results = evaluate(model, all_datasets, TRAIN_CONDITIONS, 
                                  TRAIN_REP_INDICES, DT, device)
        # Evaluate on train conditions (TEST replicates) - within-condition generalization
        within_results = evaluate(model, all_datasets, TRAIN_CONDITIONS, 
                                   TEST_REP_INDICES, DT, device)
        # Evaluate on held-out condition
        cross_results = evaluate(model, all_datasets, TEST_CONDITIONS, 
                                  list(range(N_REPLICATES)), DT, device)
        
        train_mean = np.mean([r['overall'] for r in train_results.values()])
        within_mean = np.mean([r['overall'] for r in within_results.values()])
        cross_mean = np.mean([r['overall'] for r in cross_results.values()])
        
        eval_history.append({
            'epoch': epoch + 1,
            'train': train_mean,
            'within': within_mean,
            'cross': cross_mean
        })
        
        tqdm.write(f"   Epoch {epoch+1:4d} | Loss: {loss:.4f} | "
                   f"Train: {train_mean:.3f} | Within: {within_mean:.3f} | Cross: {cross_mean:.3f}")

train_time = time.time() - start_time
print(f"\n⏱️ Training time: {train_time/60:.1f} minutes")

In [ ]:
#@title 8️⃣ Final Evaluation — The Three Tests

print("\n" + "="*70)
print("FINAL EVALUATION: THREE GENERALIZATION TESTS")
print("="*70)

# TEST 1: Within-condition generalization (train on reps 0-7, test on reps 8-9)
print("\n📊 TEST 1: Within-Condition Generalization")
print("   (Train on replicates 0-7, test on replicates 8-9 of SAME conditions)")
print("-" * 60)

test1_results = evaluate(model, all_datasets, TRAIN_CONDITIONS, TEST_REP_INDICES, DT, device)

print(f"{'Condition':<25} {'Overall':>12} {'AA Avg':>12}")
print("-" * 50)
for cond, res in test1_results.items():
    status = "✅" if res['overall'] > 0.7 else "⚠️" if res['overall'] > 0.4 else "❌"
    print(f"{status} {cond:<22} {res['overall']:>12.4f} {res['aa_avg']:>12.4f}")

test1_mean = np.mean([r['overall'] for r in test1_results.values()])
print("-" * 50)
print(f"{'MEAN':<25} {test1_mean:>12.4f}")

# TEST 2: Cross-condition generalization (held-out condition)
print("\n📊 TEST 2: Cross-Condition Generalization")
print("   (Test on HELD-OUT condition never seen during training)")
print("-" * 60)

test2_results = evaluate(model, all_datasets, TEST_CONDITIONS, list(range(N_REPLICATES)), DT, device)

print(f"{'Condition':<25} {'Overall':>12} {'AA Avg':>12}")
print("-" * 50)
for cond, res in test2_results.items():
    status = "✅" if res['overall'] > 0.7 else "⚠️" if res['overall'] > 0.4 else "❌"
    print(f"{status} {cond:<22} {res['overall']:>12.4f} {res['aa_avg']:>12.4f}")

test2_mean = np.mean([r['overall'] for r in test2_results.values()])

# TEST 3: Perturbation Interpolation
print("\n📊 TEST 3: Perturbation Interpolation")
print("   (Test on NOVEL perturbation vectors not seen during training)")
print("-" * 60)

# Generate data for interpolated perturbations
def generate_interpolated_data(perturb_name, perturb_vector, base_condition='heat_shock'):
    """Generate plausible data for an interpolated perturbation."""
    # Use heat shock as base and modify based on perturbation
    base_data = all_datasets[base_condition]['replicates'][0].copy()
    times = all_datasets[base_condition]['times']
    
    # Scale response based on perturbation magnitude
    temp_effect = perturb_vector[0]
    h2o2_effect = perturb_vector[1]
    
    # Modify response intensity
    modified_data = 1.0 + (base_data - 1.0) * (0.5 + 0.5 * abs(temp_effect) + 0.5 * h2o2_effect)
    
    return modified_data, times

test3_results = {}
for interp_name, interp_data in PERTURBATION_INTERPOLATIONS.items():
    # Create synthetic test data for this interpolation
    test_data, times = generate_interpolated_data(interp_name, interp_data['vector'])
    
    # Prepare inputs
    data_t = torch.tensor(test_data[np.newaxis, :, :], dtype=torch.float32, device=device)
    perturb_t = torch.tensor([interp_data['vector']], dtype=torch.float32, device=device)
    times_t = torch.tensor(times, dtype=torch.float32, device=device)
    
    M0 = data_t[:, 0, :]
    early_traj = data_t[:, :N_CONTEXT_STEPS, :]
    
    model.eval()
    with torch.no_grad():
        pred = model.simulate(M0, perturb_t, early_traj, times_t, dt=DT)
    
    pred_np = pred[0].cpu().numpy()
    true_np = test_data
    
    corr = np.corrcoef(pred_np.flatten(), true_np.flatten())[0, 1]
    test3_results[interp_name] = {'overall': corr, 'description': interp_data['description']}

print(f"{'Perturbation':<25} {'Correlation':>12} {'Description'}")
print("-" * 70)
for name, res in test3_results.items():
    status = "✅" if res['overall'] > 0.5 else "⚠️" if res['overall'] > 0.2 else "❌"
    print(f"{status} {name:<22} {res['overall']:>12.4f}   {res['description']}")

test3_mean = np.mean([r['overall'] for r in test3_results.values()])
print("-" * 70)
print(f"{'MEAN':<25} {test3_mean:>12.4f}")

In [ ]:
#@title 9️⃣ Summary: V19 vs Previous Versions

print("\n" + "="*70)
print("COMPLETE COMPARISON: V15 → V19")
print("="*70)

print(f"""
┌────────────────────────────────────────────────────────────────────────┐
│                         RESULTS SUMMARY                                │
├────────────────────────────────────────────────────────────────────────┤
│                                                                        │
│  Version │ Architecture              │ Train    │ Test     │ Gap      │
│  ────────┼───────────────────────────┼──────────┼──────────┼──────────│
│  V15     │ No condition info         │ 0.87     │ 0.87*    │ 0.00     │
│          │ (same condition)          │          │ *same!   │          │
│  ────────┼───────────────────────────┼──────────┼──────────┼──────────│
│  V16     │ Condition embeddings      │ ~0.1     │ 0.03     │ ~0.07    │
│          │ (simultaneous training)   │          │          │          │
│  ────────┼───────────────────────────┼──────────┼──────────┼──────────│
│  V17     │ No condition info         │ 0.85     │ 0.02     │ 0.83     │
│          │ (curriculum learning)     │          │          │          │
│  ────────┼───────────────────────────┼──────────┼──────────┼──────────│
│  V18     │ Random baseline           │ N/A      │ -0.04    │ N/A      │
│          │ (zero-shot)               │          │          │          │
│  ────────┼───────────────────────────┼──────────┼──────────┼──────────│
│  V19     │ Perturbation encoding     │          │          │          │
│          │ + Context encoder         │          │          │          │
│          │ + Proper train/test       │          │          │          │
│  ────────┼───────────────────────────┼──────────┼──────────┼──────────│
│          │ TEST 1: Within-condition  │  ---     │ {test1_mean:.4f}   │          │
│          │ TEST 2: Cross-condition   │  ---     │ {test2_mean:.4f}   │          │
│          │ TEST 3: Interpolation     │  ---     │ {test3_mean:.4f}   │          │
│                                                                        │
└────────────────────────────────────────────────────────────────────────┘
""")

# Interpretation
print("\n📊 INTERPRETATION:")
print("-" * 60)

if test1_mean > 0.7:
    print("✅ TEST 1 PASSED: Within-condition generalization works!")
    print("   Model can predict unseen replicates of trained conditions.")
else:
    print("⚠️ TEST 1: Within-condition needs improvement")

if test2_mean > 0.4:
    print("✅ TEST 2 PASSED: Cross-condition generalization works!")
    print("   Model can predict held-out stress types!")
elif test2_mean > 0.1:
    print("⚠️ TEST 2 PARTIAL: Some cross-condition transfer")
else:
    print("❌ TEST 2 FAILED: Cross-condition still not working")

if test3_mean > 0.4:
    print("✅ TEST 3 PASSED: Perturbation interpolation works!")
    print("   Model can handle novel perturbation vectors!")
else:
    print("⚠️ TEST 3: Interpolation needs more work")

# Did the fixes help?
print("\n" + "="*70)
print("DID THE FIXES WORK?")
print("="*70)

v17_cross = 0.02
v18_baseline = -0.04

print(f"\n   V17 cross-condition:  {v17_cross:.3f}")
print(f"   V18 zero-shot:        {v18_baseline:.3f}")
print(f"   V19 cross-condition:  {test2_mean:.3f}")
print(f"\n   Improvement over V17: {test2_mean - v17_cross:+.3f}")
print(f"   Improvement over V18: {test2_mean - v18_baseline:+.3f}")

if test2_mean > v17_cross + 0.1:
    print("\n🎉 YES! The fixes significantly improved cross-condition generalization!")
elif test2_mean > v17_cross:
    print("\n📈 Marginal improvement. Fixes helped but more work needed.")
else:
    print("\n🔴 Fixes did not improve cross-condition. Need different approach.")

In [ ]:
#@title 🔟 Visualize Results

fig = plt.figure(figsize=(16, 12))

# 1. Training curves with evaluation checkpoints
ax1 = fig.add_subplot(2, 2, 1)
ax1.semilogy(losses, alpha=0.5, label='Training Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Progress', fontsize=12)
ax1.grid(True, alpha=0.3)
ax1.legend()

# Overlay evaluation metrics
ax1b = ax1.twinx()
epochs_eval = [e['epoch'] for e in eval_history]
ax1b.plot(epochs_eval, [e['train'] for e in eval_history], 'g-o', label='Train Corr', ms=4)
ax1b.plot(epochs_eval, [e['within'] for e in eval_history], 'b-s', label='Within-Cond Corr', ms=4)
ax1b.plot(epochs_eval, [e['cross'] for e in eval_history], 'r-^', label='Cross-Cond Corr', ms=4)
ax1b.set_ylabel('Correlation')
ax1b.legend(loc='center right')
ax1b.set_ylim(-0.2, 1.0)

# 2. Bar chart of three tests
ax2 = fig.add_subplot(2, 2, 2)
tests = ['Within-Condition\n(Test 1)', 'Cross-Condition\n(Test 2)', 'Interpolation\n(Test 3)']
values = [test1_mean, test2_mean, test3_mean]
colors = ['green' if v > 0.5 else 'orange' if v > 0.2 else 'red' for v in values]
bars = ax2.bar(tests, values, color=colors, edgecolor='black', alpha=0.7)
ax2.axhline(y=0.02, color='gray', linestyle='--', label='V17 baseline (0.02)')
ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax2.set_ylabel('Correlation')
ax2.set_title('V19: Three Generalization Tests', fontsize=12)
ax2.set_ylim(-0.2, 1.0)
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar, val in zip(bars, values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
             f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')

# 3. Within-condition details
ax3 = fig.add_subplot(2, 2, 3)
conds = list(test1_results.keys())
corrs = [test1_results[c]['overall'] for c in conds]
bars = ax3.barh(conds, corrs, color='steelblue', edgecolor='black', alpha=0.7)
ax3.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
ax3.axvline(x=0.7, color='green', linestyle='--', alpha=0.5, label='Good (0.7)')
ax3.set_xlabel('Correlation')
ax3.set_title('Test 1: Within-Condition (held-out replicates)', fontsize=12)
ax3.set_xlim(-0.2, 1.0)
ax3.legend()
ax3.grid(axis='x', alpha=0.3)

# 4. Example trajectories from cross-condition test
ax4 = fig.add_subplot(2, 2, 4)
cond = TEST_CONDITIONS[0]
ds = all_datasets[cond]
perturb_t = torch.tensor([ds['perturbation']], dtype=torch.float32, device=device)
times = ds['times']
times_t = torch.tensor(times, dtype=torch.float32, device=device)

data_t = torch.tensor(ds['replicates'][0:1], dtype=torch.float32, device=device)
M0 = data_t[:, 0, :]
early_traj = data_t[:, :N_CONTEXT_STEPS, :]

model.eval()
with torch.no_grad():
    pred = model.simulate(M0, perturb_t, early_traj, times_t, dt=DT)

pred_np = pred[0].cpu().numpy()
true_np = ds['replicates'][0]

plot_mets = ['glucose-6-P', 'pyruvate', 'ATP', 'glutamate', 'citrate']
for met in plot_mets:
    if met in metabolite_names:
        idx = metabolite_names.index(met)
        ax4.plot(times * 60, true_np[:, idx], 'o-', label=f'{met} (true)', lw=2, ms=4)
        ax4.plot(times * 60, pred_np[:, idx], 's--', label=f'{met} (pred)', lw=1.5, ms=3, alpha=0.7)

cross_corr = test2_results[cond]['overall']
ax4.set_xlabel('Time (min)')
ax4.set_ylabel('Concentration')
ax4.set_title(f'Test 2: Cross-Condition ({cond}, r={cross_corr:.3f})', fontsize=12)
ax4.legend(fontsize=7, ncol=2)
ax4.grid(True, alpha=0.3)

plt.suptitle(f'Dark Manifold V19 — Complete Fix\nWithin: {test1_mean:.3f} | Cross: {test2_mean:.3f} | Interpolation: {test3_mean:.3f}',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('v19_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n💾 Saved: v19_results.png")

In [ ]:
#@title 1️⃣1️⃣ Save Results

save_dict = {
    'version': 'V19',
    'fixes_applied': [
        'FIX 1: Perturbation encoding as explicit input',
        'FIX 2: Multiple replicates with proper train/test split',
        'FIX 3: Context encoder for adaptation'
    ],
    'config': {
        'n_epochs': N_EPOCHS,
        'hidden_dim': HIDDEN_DIM,
        'n_reactions': N_REACTIONS,
        'perturbation_dim': PERTURB_DIM,
        'context_dim': CONTEXT_DIM,
        'n_context_steps': N_CONTEXT_STEPS,
        'n_replicates': N_REPLICATES,
        'train_replicates': TRAIN_REPLICATES,
        'test_replicates': TEST_REPLICATES
    },
    'results': {
        'test1_within_condition': {
            'mean': float(test1_mean),
            'per_condition': {k: float(v['overall']) for k, v in test1_results.items()}
        },
        'test2_cross_condition': {
            'mean': float(test2_mean),
            'per_condition': {k: float(v['overall']) for k, v in test2_results.items()}
        },
        'test3_interpolation': {
            'mean': float(test3_mean),
            'per_perturbation': {k: float(v['overall']) for k, v in test3_results.items()}
        }
    },
    'comparison': {
        'v17_cross_condition': 0.02,
        'v18_zero_shot': -0.04,
        'v19_cross_condition': float(test2_mean),
        'improvement_over_v17': float(test2_mean - 0.02),
        'improvement_over_v18': float(test2_mean - (-0.04))
    },
    'train_time_minutes': train_time / 60,
    'eval_history': eval_history
}

with open('v19_results.json', 'w') as f:
    json.dump(save_dict, f, indent=2)
print("💾 Saved: v19_results.json")

torch.save({
    'model_state_dict': model.state_dict(),
    'config': save_dict['config'],
    'results': save_dict['results']
}, 'v19_model.pt')
print("💾 Saved: v19_model.pt")

# Download
from google.colab import files
files.download('v19_results.json')
files.download('v19_results.png')
files.download('v19_model.pt')

# 📌 V19 Summary: The Complete Fix

## Three Problems → Three Solutions

| Problem | Solution | Implementation |
|---------|----------|----------------|
| Model doesn't know stress type | **Perturbation encoding** | 5D vector [temp, H2O2, glucose, lactose, growth] |
| Testing extrapolation not interpolation | **Multiple replicates** | 10 replicates/condition, 8 train / 2 test |
| No adaptation mechanism | **Context encoder** | Reads first 3 timepoints to adapt |

## V19 Architecture

```
Perturbation Vector ──► Perturbation Encoder ──┐
                                               ├──► Condition Embedding ──► Dynamics
Early Timepoints ─────► Context Encoder ───────┘
```

## Three Generalization Tests

1. **Within-Condition**: Train on reps 0-7, test on reps 8-9 of same conditions
2. **Cross-Condition**: Train on 5 conditions, test on held-out 6th
3. **Interpolation**: Test on novel perturbation vectors (e.g., mild heat = average of normal and heat shock)

## Key Insight

The model now has the information it needs to make predictions:
- **What stress?** → Perturbation vector
- **This specific experiment?** → Context from early timepoints
- **General metabolic physics?** → Learned stoichiometry and flux networks